# M12.1a — Train-only Mahalanobis weighted MSE

## Hypothesis and frozen experiment
Hypothesis: modestly upweighting peripheral **training labels** reduces regression toward the mean for `chirp_mass`, `total_mass`, and `chi_eff`, measured by prediction slope distance from 1, both signed-tail bias magnitudes, and peripheral RMSE/MAE, while preserving global accuracy. This is preparation only: no training, inference, metrics, or numerical checks have been executed here. No improvement is claimed.

Freeze the final M11 G1 dataset, 400k/40k/30k/30k row memberships, seed 123, preprocessing, input z-score, saved train-only label scalers, architecture, AdamW parameters, batching, and historical evaluation. Only sample weighting changes. M11 files/artifacts remain immutable. M12.2 balanced sampling, alternative losses, architecture changes, calibration changes, and real-event inference are outside scope.

Use [M12.0](M12_0_frozen_baseline_and_acceptance.ipynb) and its frozen `outputs/m12_0_baseline_contract.json`. Candidate selection must use validation, not repeated selection against frozen test results. Do not optimize on the nine real events. Numerical guardrails and meaningful improvement thresholds remain provisional pending paired uncertainty assessment; preregister them and the label tradeoff rule before inspecting candidate outcomes or launching expensive training.

## Existing implementation and reduction
Entry point: `scripts/train_cnn_hdf5.py`; trainer: `src/models/train.py`; final G1 loader: `src/models/hdf5_batch_dataset.py`. Training currently standardizes targets using `(y - y_mean) / (y_std + 1e-8)`; this is unchanged. The old JSON `loss: MSELoss` remains intact. The optional `training.sample_weighting` block enables weighting; absence preserves the original two-tensor batches and `nn.MSELoss()` path. Optional weighting supports only the HDF5 batch modes and rejects other modes explicitly.

Exact M12.0 label-space reference: physical training-label mean; population standard deviation (ddof=0); covariance (ddof=1); scaled covariance C; inverse if rank=3, otherwise pseudoinverse with rcond=1e-12. Distances are sqrt(zᵀ C⁻¹ z). Fit the empirical CDF on the same sorted 400000 train rows: `q_i = searchsorted(sorted_train_distances, d_i, side='right') / N_train`. Ties receive the same percentile. No latent features, validation, calibration, test, or real labels enter either fit.

Raw weights: `w_i = 1 + 0.5*q_i`. Normalize **once over all training rows**, `a_i = w_i / mean_train(w)`. With standardized residuals, per-sample loss is `ell_i = mean_j((pred_ij - y_ij)**2)` and batch loss is `mean_i(a_i * ell_i)`. Do not divide by the current batch's weight sum: that would change the objective under the frozen block batching. Mean-one normalization avoids changing the overall loss scale systematically. This is a common sample weight for all three labels, not label weighting. Alpha=0 yields unit weights and the same mathematical MSE and gradients; floating-point reduction order can differ slightly, so the toy check uses numerical tolerance rather than claiming bitwise-identical training.

The shared `sample_weighting.py` helper reads only physical train labels once per process/run, without signals or RNG. It returns a float32 lookup by original G1 row ID; nontrain rows are NaN to reject leakage. The loader selects weights with the exact `batch_indices` after within-batch permutation and HDF5 order restoration. Worker partitioning, shuffling, drop_last and seeds remain unchanged. The frozen drop_last policy omits the final 128 sorted train rows per epoch; the mean-one reference still uses all 400000 train rows, without changing that historical behavior.

Validation batches contain no weights. Validation loss remains sample-averaged unweighted standardized MSE. Checkpoint improvement remains strict `val_loss < best_val_loss`; early stopping and AdamW are untouched. Weighted `train_loss` is a different diagnostic from historical unweighted train loss and must be labeled accordingly. Full weighting statistics and hashes are stored in checkpoint `model_config.sample_weighting`; inference architecture kwargs remain unchanged.

In [1]:
from pathlib import Path
import sys, json, copy
import numpy as np
import pandas as pd
import torch
from IPython.display import display

ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
            if (p / "src/paths.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.paths import resolve_data_root, resolve_processed_artifact
from src.models.sample_weighting import (
    mahalanobis_sample_weights, load_train_sample_weights, sample_weighted_mse, array_hash)
from src.models.hdf5_batch_dataset import HDF5BatchIterableDataset

M11_CONFIG_PATH = ROOT / "configs/experiments/train_m11_final_G1_500k.json"
M12_CONFIG_PATH = ROOT / "configs/experiments/train_m12_1a_weighted_mse.json"

m11 = json.loads(M11_CONFIG_PATH.read_text())
m12 = json.loads(M12_CONFIG_PATH.read_text())

# Reutilizar la resolución de rutas del baseline M11/M12.0
DATA_ROOT = resolve_data_root(config_data_root=m11["data_root"])

print("Repository root:", ROOT)
print("Data root:", DATA_ROOT)

assert DATA_ROOT.is_dir(), f"Data root not found: {DATA_ROOT}"

LABELS = ["chirp_mass", "total_mass", "chi_eff"]

expected = copy.deepcopy(m11)
expected.pop("data_root", None)
expected["training"]["sample_weighting"] = dict(
    method="train_label_mahalanobis", alpha=0.5, normalization="train_mean_one")
expected["outputs"]["checkpoint_tag"] = "M12_1a_G1_mahalanobis_alpha0p5_meanone_seed123"

assert m12 == expected, "Only weighting/output tag and portable root resolution may differ"
assert m12["training"]["loss"] == "MSELoss"

paths = {k:resolve_processed_artifact(data_root=DATA_ROOT,
    dataset_id=m12["dataset"]["dataset_id"], file_name=m12["dataset"][k], allow_legacy_flat=False)
    for k in ("dataset_file", "split_file", "label_stats_file")}

contract_path = ROOT / "notebooks/m12/outputs/m12_0_baseline_contract.json"

contract = json.loads(contract_path.read_text())

assert contract["schema_version"] == 1 and contract["dataset_id"] == "M11_final_G1_500k"
assert contract["labels"] == LABELS

print("Config contract matches; training has not been invoked.")

Repository root: /afs/ciemat.es/user/v/vserrano/Desktop/gw/Gravitational-Waves-Lab/cbc_pe
Data root: /data/vserrano/cbc_pe_data
Config contract matches; training has not been invoked.


In [2]:
for name, path in paths.items():
    print(f"{name}: {path}")
    assert path.is_file(), f"Missing artifact: {path}"

print("All M12.1 data paths resolved successfully.")

dataset_file: /data/vserrano/cbc_pe_data/processed/M11_final_G1_500k/m11_final_G1_500k_trainio.h5
split_file: /data/vserrano/cbc_pe_data/processed/M11_final_G1_500k/M11_final_G1_500k_splits_train400000_val40000_cal30000_test30000_seed123.npz
label_stats_file: /data/vserrano/cbc_pe_data/processed/M11_final_G1_500k/M11_final_G1_500k_label_stats_train_only_train400000_val40000_cal30000_test30000_seed123.npz
All M12.1 data paths resolved successfully.


## Small deterministic numerical verification — manually invoke
This cell uses tiny in-memory arrays only. It checks the shared weight formula, tie convention, alpha=0 loss/gradient equivalence, manual weighted reduction, and actual loader batch alignment under shuffling and both slice/fancy indexing branches. It does not train a model, generate a dataset, or write an artifact.

In [3]:
from types import SimpleNamespace

train_toy = np.array([[1., 2., 0.], [2., 4., 1.], [4., 3., -1.],
                      [5., 8., .5], [3., 6., -.5], [1., 2., 0.]])
a, q, meta = mahalanobis_sample_weights(train_toy, alpha=0.5)
assert np.isfinite(q).all() and ((q >= 0) & (q <= 1)).all()
assert q[0] == q[-1] and q.max() == 1
np.testing.assert_allclose(a, (1 + .5*q) / (1 + .5*q).mean(), rtol=1e-7)
np.testing.assert_allclose(a.mean(), 1., rtol=1e-7)
a0, _, _ = mahalanobis_sample_weights(train_toy, alpha=0.)
assert np.array_equal(a0, np.ones(len(train_toy), dtype=np.float32))
pred = torch.tensor([[1., -2., 3.], [2., 1., -1.], [0., 3., 2.],
                     [1., 0., 4.], [-1., 2., 1.], [2., 0., -2.]], requires_grad=True)
target = torch.zeros_like(pred)
unit_loss = sample_weighted_mse(pred, target, torch.from_numpy(a0))
base_loss = torch.nn.MSELoss()(pred, target)
assert torch.allclose(unit_loss, base_loss, rtol=1e-6, atol=1e-7)
g_unit = torch.autograd.grad(unit_loss, pred, retain_graph=True)[0]
g_base = torch.autograd.grad(base_loss, pred)[0]
assert torch.allclose(g_unit, g_base, rtol=1e-6, atol=1e-7)
weighted = sample_weighted_mse(pred, target, torch.from_numpy(a))
manual = sum(float(a[i]) * sum(float(v)**2 for v in pred[i].detach()) / 3
             for i in range(len(a))) / len(a)
assert np.isclose(weighted.item(), manual, rtol=1e-6)

row_ids = np.array([1, 3, 5, 7, 9, 11])
lookup = np.full(12, np.nan, dtype=np.float32)
lookup[row_ids] = a
# Minimal in-memory stand-in for HDF5 arrays; invoke the actual loader method.
toy_h5 = {"X": np.arange(12, dtype=np.float32).reshape(12, 1, 1),
          "y": np.repeat(np.arange(12, dtype=np.float32)[:, None], 3, axis=1)}
for overread in (1., 4.):
    batch_source = SimpleNamespace(batch_size=3, sorted_indices=row_ids, drop_last=False,
        shuffle_within_batch=True, seed=123, epoch=2, max_slice_overread=overread,
        x_key="X", y_key="y", input_normalization=None, y_mean=None, y_std=None,
        sample_weights_by_row=lookup)
    for batch_id in (1, 0):
        xb, yb, wb = HDF5BatchIterableDataset._read_batch(batch_source, toy_h5, batch_id)
        observed_ids = xb[:, 0, 0].numpy().astype(int)
        assert np.array_equal(wb.numpy(), lookup[observed_ids])
        assert np.array_equal(yb[:, 0].numpy(), observed_ids)
print("Toy weighting, alpha=0, gradients, and shuffled row alignment verified.")

Toy weighting, alpha=0, gradients, and shuffled row alignment verified.


## Train-only preparation and reference agreement — manually invoke
Read only saved metadata and training labels. Compute the reference once, outside batching. Compare its statistics and hashes with the frozen M12.0 geometry. This section writes nothing and does not load signals. It does not access M12.0 test metric values for selection. Exact distance hashes can depend on numerical-library/platform details; a mismatch requires investigation, not silently replacing the frozen reference.

In [4]:
with np.load(paths["split_file"], allow_pickle=False) as z:
    split_ids = {s:z[f"{s}_idx"].copy() for s in ("train", "val", "cal", "test")}
    assert int(z["seed"].item()) == 123
assert {s:len(v) for s,v in split_ids.items()} == dict(train=400000,val=40000,cal=30000,test=30000)
assert np.array_equal(np.sort(np.concatenate(list(split_ids.values()))), np.arange(500000))
with np.load(paths["label_stats_file"], allow_pickle=False) as z:
    assert z["label_names"].tolist() == LABELS
    assert np.array_equal(np.sort(z["train_idx"]), np.sort(split_ids["train"]))
    y_mean, y_std = z["y_mean"].copy(), z["y_std"].copy()
weights_by_row, train_q, weight_metadata = load_train_sample_weights(
    paths["dataset_file"], split_ids["train"], alpha=m12["training"]["sample_weighting"]["alpha"])
reference = contract["geometry"]
for key in ("train_indices_sha256", "reference_distance_sha256", "n_train", "rank", "pinv_rcond"):
    assert weight_metadata[key] == reference[key], f"Frozen geometry mismatch: {key}"
for key in ("mean", "std_ddof0", "covariance_ddof1", "scaled_covariance", "precision"):
    np.testing.assert_allclose(weight_metadata[key], reference[key], rtol=1e-12, atol=1e-12)
train_weights = weights_by_row[np.sort(split_ids["train"])]
assert np.isfinite(train_weights).all() and (train_weights > 0).all()
np.testing.assert_allclose(train_weights.mean(), 1., rtol=1e-6)
for s in ("val", "cal", "test"):
    assert np.isnan(weights_by_row[split_ids[s]]).all(), "Nontrain weights must be unset"
display(pd.DataFrame([dict(alpha=.5, n=len(train_weights), q_min=train_q.min(), q_max=train_q.max(),
    weight_min=train_weights.min(), weight_max=train_weights.max(), weight_mean=train_weights.mean(),
    covariance_rank=weight_metadata["rank"], covariance_condition=weight_metadata["condition_number"])]))

,alpha,n,q_min,q_max,weight_min,weight_max,weight_mean,covariance_rank,covariance_condition
0,0.5,400000,0.000003,1.0,0.800001,1.199999,1.0,3,47.237791


## Fixed M11 validation predictions — reuse before requesting inference
Targeted archive-member inspection found `results/M11_final_G1_500k/m11_final_G1_500k_val_predictions_embeddings.npz` with `pred_val`, `y_val`, `idx_val`, `y_mean`, and `y_std`. The older cal/test archive does not contain validation predictions. Presence is verified; numerical contents and provenance are deliberately not evaluated during implementation. The following cell validates the existing trusted saved archive without loading embeddings, predicting, or fitting anything. It does not compute candidate metrics.

If missing or incompatible on another machine, stop. The non-training solution is an explicitly authorized **single validation-only inference pass** with the frozen M11 checkpoint using existing `scripts/predict_cnn_hdf5.py`, the unchanged G1 split/scaler/input normalization and `prediction.splits=["val"]`. Preserve `idx_val` (the batch loader emits sorted IDs), set `output.overwrite=false`, and write a separate archive. Do not retrain M11 or use cal/test predictions as a substitute. No inference command is launched here.

In [5]:
val_path = DATA_ROOT / "results/M11_final_G1_500k/m11_final_G1_500k_val_predictions_embeddings.npz"
if not val_path.is_file():
    raise FileNotFoundError(f"Missing fixed M11 validation predictions: {val_path}; request inference separately")
checkpoint_name = ("M11_final_G1_500k_SimpleCNN_ResidualDilated_"
    + m11["outputs"]["checkpoint_tag"] + "_MSELoss_seed123_checkpoint.pt")
# Trusted project NPZ only: model/input provenance is stored as object dictionaries.
with np.load(val_path, allow_pickle=True) as z:
    assert z["label_names"].tolist() == LABELS
    assert z["available_splits"].tolist() == ["val"]
    assert Path(z["checkpoint_file"].item()).name == checkpoint_name
    for saved, local in (("dataset_path", "dataset_file"), ("split_path", "split_file"),
                         ("label_stats_path", "label_stats_file")):
        assert Path(z[saved].item()).name == paths[local].name
    assert z["input_normalization"].item() == m11["input_normalization"]
    assert z["model_config"].item()["dataset_id"] == "M11_final_G1_500k"
    assert np.array_equal(z["y_mean"], y_mean) and np.array_equal(z["y_std"], y_std)
    val_idx = z["idx_val"].copy()
    assert np.issubdtype(val_idx.dtype, np.integer)
    assert np.array_equal(np.sort(val_idx), np.sort(split_ids["val"]))
    val_true_std, val_pred_std = z["y_val"].copy(), z["pred_val"].copy()
    assert val_true_std.shape == val_pred_std.shape == (40000, 3)
    assert np.isfinite(val_true_std).all() and np.isfinite(val_pred_std).all()
# Check target pairing with physical HDF5 labels, in the archive's original order.
import h5py
order = np.argsort(val_idx)
with h5py.File(paths["dataset_file"], "r") as h:
    assert (h["split"].asstr()[np.sort(val_idx)] == "val").all()
    for start in range(0, len(val_idx), 4096):
        pos = order[start:start+4096]
        expected_y = h["y"][val_idx[pos]].astype(np.float32)
        expected_y = (expected_y - y_mean) / (y_std + 1e-8)
        np.testing.assert_allclose(val_true_std[pos], expected_y, rtol=2e-6, atol=2e-5)
print("Saved fixed M11 validation archive passed provenance and target-pairing checks.")

Saved fixed M11 validation archive passed provenance and target-pairing checks.


## Output isolation and manual run policy
The separate config is `configs/experiments/train_m12_1a_weighted_mse.json`. Its scientific fields equal frozen M11 except `training.sample_weighting`. The output checkpoint tag is `M12_1a_G1_mahalanobis_alpha0p5_meanone_seed123`, isolating both checkpoint and history names under the existing final G1 artifact directories. The machine-specific `data_root` is omitted: use `CBC_PE_DATA_ROOT` or `--data-root` to point at the **same** frozen root. No dataset or output directory is renamed. The weighted entry point refuses existing checkpoint/history destinations before training starts.

For manual preparation, run imports/config assertions, the toy verification, train-reference agreement, then saved-validation checks in order. Do not Run All. No training command is part of this notebook. A later authorized training launch uses the existing `scripts/train_cnn_hdf5.py --config configs/experiments/train_m12_1a_weighted_mse.json` with the frozen data root; it is not authorized by executing preparation cells.

Pending scientific decisions: numerical global/tail/peripheral degradation tolerances, minimum meaningful improvements, paired uncertainty procedure (source grouping remains unresolved beyond row IDs), and how to handle tradeoffs among the three labels. Compare candidate and M11 on the same validation IDs in physical units using M12.0 diagnostic definitions; validation truth quantiles define diagnostic tails only, while Mahalanobis geometry/CDF remain train-only. Preregister the choice before candidate outcomes. The frozen test is reserved for a final audit; real events remain a separate final descriptive comparison.

Checkpoint filenames and provenance checks do not independently prove checkpoint byte identity. The runtime reference/hash checks, saved validation checks, and toy verification remain unexecuted. Python 3.10 syntax is checked statically; the implementation uses APIs available in historical PyTorch 1.9.1 (ordinary tensor arithmetic and DataLoader), without new dependencies. No runtime compatibility, scientific improvement, or training reproducibility result is claimed.